# End-to-end evaluation of the scoring pipeline (issue #10)

Starts from the raw daily table `data/synthetic_cdr_daily_agg.csv` and the episode file
`data/synthetic_cdr_labels.csv`, and scores through `src.risk_scoring.score`, the same code the API
uses. The saved `X_*_v2` files are read only once, in part 1, to check that the pipeline reproduces
the probabilities of `xgboost_final.ipynb`. The SIM split is the one used by `xgboost_final.ipynb`
(700 train, 150 validation, 150 test SIMs, read from the `y_*` files).

Nothing here changes a model, an artifact or the gate. Part 6 refits temporary models in memory only.
Everything is logged to the MLflow experiment `simguard-stage5-evaluation`.

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import mlflow
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
if not os.path.isdir("data") and os.path.isdir(os.path.join("..", "data")):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
ART = Path("artifacts").resolve()
os.environ["ML_ARTIFACT_DIR"] = str(ART)
from src import risk_scoring as rs
from src import feature_engineering as fe

SEED = 42
SIM_START = pd.Timestamp("2025-01-01")
CLEAR, FLAG = rs.GATE["clear_below"], rs.GATE["flag_above"]
CATS = ["ghost_usage", "unauthorized_sharing", "data_reselling", "inactive_billing"]
pd.set_option("display.width", 200)
TIMES = {}

mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("simguard-stage5-evaluation")
if mlflow.active_run():
    mlflow.end_run()
run = mlflow.start_run(run_name="end_to_end_42col")
mlflow.set_tags({"issue": "10", "model": "xgboost_triage.joblib (42 columns)", "gate": f"{CLEAR}/{FLAG}"})

def log_table(df, name):
    mlflow.log_text(df.to_csv(), f"tables/{name}.csv")

print("CWD:", os.getcwd(), "| gate", CLEAR, FLAG, "| xgboost", xgb.__version__)

C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/02 16:42:24 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\ADMIN\Desktop\projects\simguard\ml-service\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


CWD: C:\Users\ADMIN\Desktop\projects\simguard\ml-service | gate 0.1 0.9 | xgboost 3.4.1


## 0. Raw data, labels and the SIM split

In [2]:
t0 = time.perf_counter()
daily = pd.read_csv("data/synthetic_cdr_daily_agg.csv", parse_dates=["date"])
episodes = pd.read_csv("data/synthetic_cdr_labels.csv")
episodes["start_dt"] = SIM_START + pd.to_timedelta(episodes["start_day"], unit="D")
episodes["end_dt"] = SIM_START + pd.to_timedelta(episodes["end_day"], unit="D")

# Labels per SIM-day, built from the episodes as in feature_engineering.ipynb cell 2.
labels = daily[["sim_id", "date"]].copy()
labels["is_anomalous"] = 0
labels["misuse_category"] = "normal"
for e in episodes.itertuples(index=False):
    m = (labels["sim_id"] == e.sim_id) & (labels["date"] >= e.start_dt) & (labels["date"] <= e.end_dt)
    labels.loc[m, "is_anomalous"] = 1
    labels.loc[m, "misuse_category"] = e.category
labels = labels.sort_values(["sim_id", "date"]).reset_index(drop=True)

split_sims = {}
saved = []
for name in ("train", "val", "test"):
    y = pd.read_parquet(f"data/y_{name}.parquet")
    split_sims[name] = set(y["sim_id"].unique())
    saved.append(y)
saved = pd.concat(saved).sort_values(["sim_id", "date"]).reset_index(drop=True)
assert (saved[["sim_id", "date"]].values == labels[["sim_id", "date"]].values).all()
assert (saved["is_anomalous"].values == labels["is_anomalous"].values).all()
assert (saved["misuse_category"].values == labels["misuse_category"].values).all()
assert not (split_sims["train"] & split_sims["val"]) and not (split_sims["val"] & split_sims["test"])

# Split as in feature_engineering.ipynb cell 6: shuffle the sorted SIM ids with default_rng(seed).
ALL_SIMS = np.sort(daily["sim_id"].unique())
def make_split(seed):
    s = ALL_SIMS.copy()
    np.random.default_rng(seed).shuffle(s)
    return {"train": set(s[:700]), "val": set(s[700:850]), "test": set(s[850:])}
assert make_split(42) == split_sims, "split function does not reproduce the saved split"

print(f"daily rows {len(daily):,} | SIMs {len(ALL_SIMS)} | episodes {len(episodes)} on {episodes['sim_id'].nunique()} SIMs")
print("labels rebuilt from the episode file equal the saved y files: True")
sim_cat = episodes.groupby("sim_id")["category"].agg(lambda s: ",".join(sorted(set(s))))
print("SIMs with more than one category:", int(sim_cat.str.contains(",").sum()))
counts = []
for name in ("train", "val", "test"):
    lab = labels[labels["sim_id"].isin(split_sims[name])]
    ep = episodes[episodes["sim_id"].isin(split_sims[name])]
    row = {"split": name, "SIMs": len(split_sims[name]), "anomalous SIMs": ep["sim_id"].nunique(),
           "anomalous rows": int(lab["is_anomalous"].sum())}
    for c in CATS:
        row[f"{c} rows"] = int((lab["misuse_category"] == c).sum())
        row[f"{c} SIMs"] = ep.loc[ep["category"] == c, "sim_id"].nunique()
    counts.append(row)
counts = pd.DataFrame(counts).set_index("split")
print(counts.T.to_string())
log_table(counts, "counts_per_split")
TIMES["0 data"] = time.perf_counter() - t0

daily rows 90,000 | SIMs 1000 | episodes 175 on 162 SIMs
labels rebuilt from the episode file equal the saved y files: True
SIMs with more than one category: 10
split                      train  val  test
SIMs                         700  150   150
anomalous SIMs               102   37    23
anomalous rows              1292  524   361
ghost_usage rows             447  150    80
ghost_usage SIMs              24   10     8
unauthorized_sharing rows    403  217   235
unauthorized_sharing SIMs     39   18    10
data_reselling rows          293  101    24
data_reselling SIMs           31    7     4
inactive_billing rows        149   56    22
inactive_billing SIMs         13    5     3


## 1. Pipeline check: `score()` equals the notebook probabilities

In [3]:
t0 = time.perf_counter()
model = joblib.load(ART / "xgboost_triage.joblib")
cols42 = json.load(open(ART / "xgboost_feature_columns.json"))
scored, details = {}, {}
for name in ("val", "test"):
    d = daily[daily["sim_id"].isin(split_sims[name])]
    out = rs.score(d, artifact_dir=ART).sort_values(["sim_id", "date"]).reset_index(drop=True)
    # Same steps split open, to also get the Isolation Forest score and the cohort columns.
    feats = fe.build_features(d, ART)
    scaled = fe.scale_features(feats.reset_index(drop=True), ART)
    out2, det = rs.score_features(scaled, ART, return_details=True)
    assert (out2["probability"].values == out["probability"].values).all()
    det["scaled"] = scaled
    lab = labels[labels["sim_id"].isin(split_sims[name])].reset_index(drop=True)
    assert (lab[["sim_id", "date"]].values == out[["sim_id", "date"]].values).all()
    out = pd.concat([out, lab[["is_anomalous", "misuse_category"]]], axis=1)
    scored[name], details[name] = out, det

    Xv2 = pd.read_parquet(f"data/X_{name}_v2.parquet").sort_values(["sim_id", "date"]).reset_index(drop=True)
    assert (Xv2[["sim_id", "date"]].values == out[["sim_id", "date"]].values).all()
    p_nb = model.predict_proba(Xv2[cols42])[:, 1]
    diff = np.abs(p_nb - out["probability"].values)
    same_bucket = (rs.gate(p_nb) == out["bucket"].values).mean()
    print(f"{name}: rows {len(out):,} | max |score() - notebook| = {diff.max():.3e} | "
          f"mean {diff.mean():.3e} | same bucket {same_bucket:.4%}")
    mlflow.log_metric(f"{name}_pipeline_max_abs_diff", float(diff.max()))
    assert diff.max() < 1e-4, "pipeline does not reproduce the notebook probabilities"
TIMES["1 pipeline check"] = time.perf_counter() - t0

val: rows 13,500 | max |score() - notebook| = 0.000e+00 | mean 0.000e+00 | same bucket 100.0000%


test: rows 13,500 | max |score() - notebook| = 0.000e+00 | mean 0.000e+00 | same bucket 100.0000%


## 2. Row-level results at the gate

In [4]:
t0 = time.perf_counter()
def prf(y, pred):
    return (precision_score(y, pred, zero_division=0), recall_score(y, pred), f1_score(y, pred))

rows, buckets, percat = [], [], []
for name, out in scored.items():
    y, p, b = out["is_anomalous"].values, out["probability"].values, out["bucket"].values
    for label, pred in (("flag vs rest", b == "flag"), ("flag + escalated vs clear", b != "clear")):
        pr, rc, f1 = prf(y, pred)
        rows.append({"split": name, "positive": label, "precision": pr, "recall": rc, "f1": f1})
    rows[-2]["AP"] = rows[-1]["AP"] = average_precision_score(y, p)
    rows[-2]["ROC AUC"] = rows[-1]["ROC AUC"] = roc_auc_score(y, p)
    for bk in ("clear", "escalated", "flag"):
        m = b == bk
        buckets.append({"split": name, "bucket": bk, "rows": int(m.sum()), "share": m.mean(),
                        "anomalies": int(y[m].sum()), "share of anomalies": y[m].sum() / y.sum(),
                        "precision": y[m].mean()})
    for c in CATS:
        m = out["misuse_category"].values == c
        percat.append({"split": name, "category": c, "anomalous rows": int(m.sum()),
                       "recall flag": (b[m] == "flag").mean(), "recall flag+escalated": (b[m] != "clear").mean(),
                       "in clear": int((b[m] == "clear").sum())})
gate_tab = pd.DataFrame(rows).set_index(["split", "positive"])
bucket_tab = pd.DataFrame(buckets).set_index(["split", "bucket"])
cat_tab = pd.DataFrame(percat).set_index(["split", "category"])
print(gate_tab.round(4).to_string()); print()
print(bucket_tab.round(4).to_string()); print()
print(cat_tab.round(4).to_string())
for t, n in ((gate_tab, "gate_metrics"), (bucket_tab, "buckets"), (cat_tab, "per_category")):
    log_table(t, n)
for name in scored:
    mlflow.log_metric(f"{name}_AP", gate_tab.loc[(name, "flag vs rest"), "AP"])
    mlflow.log_metric(f"{name}_ROC_AUC", gate_tab.loc[(name, "flag vs rest"), "ROC AUC"])
    mlflow.log_metric(f"{name}_flag_precision", gate_tab.loc[(name, "flag vs rest"), "precision"])
    mlflow.log_metric(f"{name}_flag_recall", gate_tab.loc[(name, "flag vs rest"), "recall"])
    mlflow.log_metric(f"{name}_band_share", bucket_tab.loc[(name, "escalated"), "share"])
# Cross-check against xgboost_final.ipynb (cells 23 and 25).
assert bucket_tab.loc[("val", "escalated"), "rows"] == 3893 and bucket_tab.loc[("test", "escalated"), "rows"] == 3887
assert abs(gate_tab.loc[("val", "flag vs rest"), "AP"] - 0.9254) < 5e-4
assert abs(gate_tab.loc[("test", "flag vs rest"), "AP"] - 0.9297) < 5e-4
TIMES["2 row level"] = time.perf_counter() - t0

                                 precision  recall      f1      AP  ROC AUC
split positive                                                             
val   flag vs rest                  0.9956  0.8588  0.9221  0.9254    0.962
      flag + escalated vs clear     0.1132  0.9389  0.2021  0.9254    0.962
test  flag vs rest                  0.9836  0.8283  0.8992  0.9297    0.972
      flag + escalated vs clear     0.0830  0.9640  0.1529  0.9297    0.972

                 rows   share  anomalies  share of anomalies  precision
split bucket                                                           
val   clear      9155  0.6781         32              0.0611     0.0035
      escalated  3893  0.2884         42              0.0802     0.0108
      flag        452  0.0335        450              0.8588     0.9956
test  clear      9309  0.6896         13              0.0360     0.0014
      escalated  3887  0.2879         49              0.1357     0.0126
      flag        304  0.0225        29

## 3. What each stage adds (same rows)

Scores compared: the Isolation Forest score alone, the best single cohort Z column (chosen by
validation AP), the final 42 column XGBoost, and the archived 53 column XGBoost as a reference
for the old design. The old model's inputs are rebuilt here from the pipeline features plus the
per-cycle Prophet forecast, exactly as `xgboost_triage.ipynb` did; that forecast was later found
to look ahead within the cycle (`prophet_lookahead_audit.ipynb`), which is why the design changed.
Recall at 5% FPR uses the score value that 5% of that split's normal rows exceed.

In [5]:
t0 = time.perf_counter()
arch = joblib.load(ART / "archive_53col" / "xgboost_triage.joblib")
arch_cols = json.load(open(ART / "archive_53col" / "xgboost_feature_columns.json"))
prophet = pd.read_parquet("data/prophet_features.parquet")[
    ["sim_id", "billing_cycle_num", "recommendation", "projected_overage_mb", "days_to_depletion"]].copy()
prophet["billing_cycle_num"] = prophet["billing_cycle_num"].astype(int)
REC = ["upgrade", "downgrade", "no_change", "insufficient_history"]

def old_inputs(base):
    X = base.copy()
    X["billing_cycle_num"] = ((X["date"] - SIM_START).dt.days // 30).astype(int)
    m = X.merge(prophet, on=["sim_id", "billing_cycle_num"], how="left", indicator=True, validate="many_to_one")
    unmatched = m["_merge"] == "left_only"
    for c in REC:
        m[f"prophet_rec_{c}"] = (m["recommendation"] == c).astype(int)
    m["prophet_missing_flag"] = unmatched.astype(int)
    dtd = m["days_to_depletion"].copy()
    dtd[~unmatched & dtd.isna()] = 31.0
    m["days_to_depletion"] = dtd.fillna(0.0)
    m["projected_overage_mb"] = m["projected_overage_mb"].fillna(0.0)
    assert len(m) == len(base)
    return m[arch_cols]

def recall_at_fpr(y, s, fpr=0.05):
    thr = np.quantile(s[y == 0], 1 - fpr)
    return (s[y == 1] > thr).mean(), (s[y == 0] > thr).mean()

stage_scores = {}
for name, out in scored.items():
    det = details[name]
    base = pd.concat([det["scaled"].reset_index(drop=True),
                      det["model_input"][[c for c in det["model_input"].columns if c.startswith("cohort_z_")]]], axis=1)
    p_old = arch.predict_proba(old_inputs(base))[:, 1]
    # Check against the old notebook route (X_v2 files plus the same merge).
    Xv2 = pd.read_parquet(f"data/X_{name}_v2.parquet").sort_values(["sim_id", "date"]).reset_index(drop=True)
    p_old_nb = arch.predict_proba(old_inputs(Xv2))[:, 1]
    print(f"{name}: archived model, max |pipeline - notebook route| = {np.abs(p_old - p_old_nb).max():.3e}")
    stage_scores[name] = {"isolation_forest": det["iso_anomaly_score"],
                          **{c: det["model_input"][c].values for c in det["model_input"].columns if c.startswith("cohort_z_")},
                          "xgboost_42": out["probability"].values, "xgboost_53_old": p_old}

yv = scored["val"]["is_anomalous"].values
cz = [c for c in stage_scores["val"] if c.startswith("cohort_z_")]
cz_ap = {c: average_precision_score(yv, stage_scores["val"][c]) for c in cz}
best_cz = max(cz_ap, key=cz_ap.get)
print("cohort Z columns, validation AP:", {k: round(v, 4) for k, v in cz_ap.items()}, "-> best:", best_cz)

rows = []
for name in ("val", "test"):
    y = scored[name]["is_anomalous"].values
    for label, key in (("Isolation Forest alone", "isolation_forest"), (f"cohort Z alone ({best_cz})", best_cz),
                       ("XGBoost 42 columns (final)", "xgboost_42"), ("XGBoost 53 columns (old design, reference)", "xgboost_53_old")):
        s = np.asarray(stage_scores[name][key], float)
        rec, real_fpr = recall_at_fpr(y, s)
        rows.append({"split": name, "stage": label, "AP": average_precision_score(y, s),
                     "recall at 5% FPR": rec, "actual FPR": real_fpr})
stage_tab = pd.DataFrame(rows).set_index(["split", "stage"])
print(stage_tab.round(4).to_string())
log_table(stage_tab, "stage_comparison")
for (split, stage), r in stage_tab.iterrows():
    key = stage.split(" (")[0].replace(" ", "_")
    mlflow.log_metric(f"{split}_stage_{key}_AP", r["AP"])
    mlflow.log_metric(f"{split}_stage_{key}_recall_at_5fpr", r["recall at 5% FPR"])
TIMES["3 stages"] = time.perf_counter() - t0

val: archived model, max |pipeline - notebook route| = 0.000e+00
test: archived model, max |pipeline - notebook route| = 0.000e+00
cohort Z columns, validation AP: {'cohort_z_iso_anomaly_score': 0.5364, 'cohort_z_total_mb': 0.2028, 'cohort_z_vol_zscore_7d': 0.0421, 'cohort_z_vol_zscore_14d': 0.046} -> best: cohort_z_iso_anomaly_score
                                                       AP  recall at 5% FPR  actual FPR
split stage                                                                            
val   Isolation Forest alone                       0.4633            0.5840        0.05
      cohort Z alone (cohort_z_iso_anomaly_score)  0.5364            0.6298        0.05
      XGBoost 42 columns (final)                   0.9254            0.9275        0.05
      XGBoost 53 columns (old design, reference)   0.9289            0.9313        0.05
test  Isolation Forest alone                       0.4944            0.7175        0.05
      cohort Z alone (cohort_z_iso_anomaly_score

## 4. SIM-level results

A SIM counts as flagged if it has at least one flag day in its 90 days. A SIM is anomalous if it
has at least one episode. Per category, precision compares the SIMs of that category with the
normal SIMs only (one category against normal), so a flagged normal SIM counts against every
category. Time to first flag uses the episode start dates in `synthetic_cdr_labels.csv`: days from
the episode start to the first flag day inside that episode.

In [6]:
t0 = time.perf_counter()
sim_rows, lead_rows = [], []
for name, out in scored.items():
    flagged = out.groupby("sim_id")["bucket"].apply(lambda s: (s == "flag").any())
    ep = episodes[episodes["sim_id"].isin(split_sims[name])]
    anomalous = set(ep["sim_id"])
    normal = [s for s in flagged.index if s not in anomalous]
    fp = int(flagged.loc[normal].sum())
    tp = int(flagged.loc[list(anomalous)].sum())
    sim_rows.append({"split": name, "group": "all anomalous", "SIMs": len(anomalous), "flagged": tp,
                     "recall": tp / len(anomalous), "precision": tp / (tp + fp) if tp + fp else np.nan})
    for c in CATS:
        sims_c = set(ep.loc[ep["category"] == c, "sim_id"])
        tp_c = int(flagged.loc[list(sims_c)].sum()) if sims_c else 0
        sim_rows.append({"split": name, "group": c, "SIMs": len(sims_c), "flagged": tp_c,
                         "recall": tp_c / len(sims_c) if sims_c else np.nan,
                         "precision": tp_c / (tp_c + fp) if tp_c + fp else np.nan})
    sim_rows.append({"split": name, "group": "normal (false alarms)", "SIMs": len(normal), "flagged": fp,
                     "recall": np.nan, "precision": np.nan, "per 100 normal SIMs": 100 * fp / len(normal)})
    flags = out[out["bucket"] == "flag"]
    for e in ep.itertuples(index=False):
        f = flags[(flags["sim_id"] == e.sim_id) & (flags["date"] >= e.start_dt) & (flags["date"] <= e.end_dt)]
        lead_rows.append({"split": name, "category": e.category, "duration": e.duration_days,
                          "days_to_first_flag": (f["date"].min() - e.start_dt).days if len(f) else np.nan})
sim_tab = pd.DataFrame(sim_rows).set_index(["split", "group"])
lead = pd.DataFrame(lead_rows)
lead_tab = lead.groupby(["split"]).agg(
    episodes=("duration", "size"), detected=("days_to_first_flag", lambda s: int(s.notna().sum())),
    median_days=("days_to_first_flag", "median"),
    p90_days=("days_to_first_flag", lambda s: s.quantile(0.9)))
lead_cat = lead.groupby(["split", "category"]).agg(
    episodes=("duration", "size"), detected=("days_to_first_flag", lambda s: int(s.notna().sum())),
    median_days=("days_to_first_flag", "median"), p90_days=("days_to_first_flag", lambda s: s.quantile(0.9)))
print(sim_tab.round(4).to_string()); print()
print("Days from episode start to first flag inside the episode (detected episodes only):")
print(lead_tab.to_string()); print()
print(lead_cat.to_string())
for t, n in ((sim_tab, "sim_level"), (lead_tab, "time_to_flag"), (lead_cat, "time_to_flag_by_category")):
    log_table(t, n)
for name in scored:
    mlflow.log_metric(f"{name}_sim_recall", sim_tab.loc[(name, "all anomalous"), "recall"])
    mlflow.log_metric(f"{name}_sim_precision", sim_tab.loc[(name, "all anomalous"), "precision"])
    mlflow.log_metric(f"{name}_false_alarm_sims_per_100", sim_tab.loc[(name, "normal (false alarms)"), "per 100 normal SIMs"])
    mlflow.log_metric(f"{name}_days_to_first_flag_median", lead_tab.loc[name, "median_days"])
    mlflow.log_metric(f"{name}_days_to_first_flag_p90", lead_tab.loc[name, "p90_days"])
TIMES["4 SIM level"] = time.perf_counter() - t0

                             SIMs  flagged  recall  precision  per 100 normal SIMs
split group                                                                       
val   all anomalous            37       32  0.8649     1.0000                  NaN
      ghost_usage              10       10  1.0000     1.0000                  NaN
      unauthorized_sharing     18       14  0.7778     1.0000                  NaN
      data_reselling            7        6  0.8571     1.0000                  NaN
      inactive_billing          5        5  1.0000     1.0000                  NaN
      normal (false alarms)   113        0     NaN        NaN               0.0000
test  all anomalous            23       23  1.0000     0.9200                  NaN
      ghost_usage               8        8  1.0000     0.8000                  NaN
      unauthorized_sharing     10       10  1.0000     0.8333                  NaN
      data_reselling            4        4  1.0000     0.6667                  NaN
    

## 5. Clear-threshold trade-off (flag threshold fixed at 0.90)

Read from validation; test is shown next to it only as a check. Nothing is chosen on test and the
gate in `src/risk_scoring.py` is not changed.

In [7]:
t0 = time.perf_counter()
rows = []
for thr in (0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25):
    r = {"clear threshold": thr}
    for name, out in scored.items():
        p, y = out["probability"].values, out["is_anomalous"].values
        cleared = p < thr
        band = (p >= thr) & (p <= FLAG)
        r[f"{name} cleared share"] = cleared.mean()
        r[f"{name} band rows"] = int(band.sum())
        r[f"{name} band share"] = band.mean()
        r[f"{name} anomalies cleared"] = int(y[cleared].sum())
    rows.append(r)
trade = pd.DataFrame(rows).set_index("clear threshold")
print(trade.round(4).to_string())
log_table(trade, "clear_threshold_tradeoff")
TIMES["5 trade-off"] = time.perf_counter() - t0

                 val cleared share  val band rows  val band share  val anomalies cleared  test cleared share  test band rows  test band share  test anomalies cleared
clear threshold                                                                                                                                                      
0.050                       0.0052          12978          0.9613                      0              0.0058           13118           0.9717                       0
0.075                       0.2055          10274          0.7610                      7              0.1935           10584           0.7840                       3
0.100                       0.6781           3893          0.2884                     32              0.6896            3887           0.2879                      13
0.125                       0.8718           1279          0.0947                     36              0.8830            1276           0.0945                      21
0.15

## 6. Stability over 5 other SIM splits

Seeds 1 to 5, same 700/150/150 SIM fractions, same split function as the saved split. For each
split, the scaler, the Isolation Forest, the cohort statistics and XGBoost are refitted on that
split's training SIMs only, with early stopping (patience 50) on that split's validation SIMs. The
fill value for the first-day rolling mean is the saved constant (see the README limitations).
Models stay in memory; no artifact is written. Seed 42 is run first through the same code as a
check: the scaler, Isolation Forest and cohort statistics must equal the saved artifacts. XGBoost
comes out close but not identical, because the pipeline features differ from the `X_*_v2` training
files by float noise (about 3e-7), which moves a few histogram bin edges.

In [8]:
t0 = time.perf_counter()
FEATS = fe.build_features(daily, ART).reset_index(drop=True)          # unscaled, all 90,000 rows
LAB = labels.set_index(["sim_id", "date"]).loc[list(zip(FEATS["sim_id"], FEATS["date"]))].reset_index()
assert (LAB[["sim_id", "date"]].values == FEATS[["sim_id", "date"]].values).all()
XGB_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                  eval_metric="aucpr", early_stopping_rounds=50, tree_method="hist", random_state=SEED, n_jobs=-1)

def refit_and_score(seed):
    sp = make_split(seed)
    part = {k: FEATS["sim_id"].isin(v).values for k, v in sp.items()}
    X = FEATS.copy()
    for c in fe.UNSCALED_CLIPPED_COLUMNS:
        X[c] = X[c].clip(-fe.ZSCORE_CLIP, fe.ZSCORE_CLIP)
    scaler = RobustScaler().fit(X.loc[part["train"], list(fe.SCALED_COLUMNS)])
    X[list(fe.SCALED_COLUMNS)] = scaler.transform(X[list(fe.SCALED_COLUMNS)])
    iso = IsolationForest(n_estimators=200, contamination="auto", random_state=SEED, n_jobs=-1)
    iso.fit(X.loc[part["train"], list(fe.FEATURE_COLUMNS)])
    X["iso_anomaly_score"] = -iso.decision_function(X[list(fe.FEATURE_COLUMNS)])
    tier = X[list(rs.TIER_COLUMNS)].idxmax(axis=1).str.replace("tier_", "", regex=False)
    stats = {}
    for col in rs.COHORT_SOURCE_COLUMNS:
        g = X.loc[part["train"], col].groupby(tier[part["train"]])
        med, iqr_raw = g.median(), g.quantile(0.75) - g.quantile(0.25)
        stats[col] = (med, iqr_raw)
        X[f"cohort_z_{col}"] = (X[col] - tier.map(med)) / tier.map(iqr_raw.clip(lower=rs.IQR_FLOOR))
    y = LAB["is_anomalous"].values
    ytr = y[part["train"]]
    m = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum())
    m.fit(X.loc[part["train"], cols42], ytr, eval_set=[(X.loc[part["val"], cols42], y[part["val"]])], verbose=False)
    p = m.predict_proba(X.loc[part["test"], cols42])[:, 1]
    yt = y[part["test"]]
    flag, band = p > FLAG, (p >= CLEAR) & (p <= FLAG)
    test_eps = episodes[episodes["sim_id"].isin(sp["test"])]
    return {"seed": seed, "best_iteration": int(m.best_iteration), "test anomalous rows": int(yt.sum()),
            "test anomalous SIMs": test_eps["sim_id"].nunique(),
            "AP": average_precision_score(yt, p), "flag precision": precision_score(yt, flag, zero_division=0),
            "flag recall": recall_score(yt, flag), "band share": band.mean(), "band rows": int(band.sum()),
            "anomalies in clear": int(yt[p < CLEAR].sum())}, p, {"scaler": scaler, "iso": iso, "stats": stats, "X": X, "part": part}

ref, p_ref, parts = refit_and_score(42)
# Scaler, Isolation Forest and cohort stats must equal the saved artifacts exactly.
saved_scaler = joblib.load(ART / "feature_scaler.joblib")
assert np.allclose(parts["scaler"].center_, saved_scaler.center_, atol=1e-9, rtol=0)
assert np.allclose(parts["scaler"].scale_, saved_scaler.scale_, atol=1e-9, rtol=0)
saved_iso = joblib.load(ART / "isolation_forest.joblib")
Xi = parts["X"][list(fe.FEATURE_COLUMNS)]
iso_diff = np.abs(parts["iso"].decision_function(Xi) - saved_iso.decision_function(Xi)).max()
saved_stats = json.load(open(ART / "cohort_stats.json"))
stat_diff = max(max(abs(parts["stats"][c][0][t] - saved_stats[c]["median"][t]),
                    abs(parts["stats"][c][1][t] - saved_stats[c]["iqr"][t]))
                for c in rs.COHORT_SOURCE_COLUMNS for t in saved_stats[c]["median"])
print(f"seed 42: scaler equal, Isolation Forest max diff {iso_diff:.1e}, cohort stats max diff {stat_diff:.1e}")
assert iso_diff < 1e-9 and stat_diff < 1e-6
# XGBoost: the pipeline features differ from the X_*_v2 files only by float noise (max about 3e-7 in the
# rolling std columns). With tree_method="hist" that noise moves a few bin edges, so the refit model is
# close to the saved one but not identical. Trained on the X_*_v2 files, the same code gives the saved
# model exactly (best_iteration 56). Reported, not asserted.
p_saved = scored["test"]["probability"].values
print(f"seed 42 XGBoost refit vs saved model on test: max |diff| = {np.abs(p_ref - p_saved).max():.3e}, "
      f"AP {ref['AP']:.4f} (saved 0.9297), best_iteration {ref['best_iteration']} (saved 56), "
      f"band rows {ref['band rows']} (saved 3887), flag precision {ref['flag precision']:.4f}, recall {ref['flag recall']:.4f}")
mlflow.log_metric("seed42_refit_max_abs_diff", float(np.abs(p_ref - p_saved).max()))

stab = []
for seed in (1, 2, 3, 4, 5):
    ts = time.perf_counter()
    r, _, _ = refit_and_score(seed)
    r["seconds"] = round(time.perf_counter() - ts, 1)
    stab.append(r)
    with mlflow.start_run(run_name=f"stability_seed_{seed}", nested=True):
        mlflow.log_metrics({k.replace(" ", "_"): float(v) for k, v in r.items() if k != "seed"})
        mlflow.log_param("seed", seed)
stab = pd.DataFrame(stab).set_index("seed")
print(stab.round(4).to_string())
summary = stab[["AP", "flag precision", "flag recall", "band share", "anomalies in clear"]].agg(["mean", "std"])
print(); print(summary.round(4).to_string())
log_table(stab, "stability_per_seed"); log_table(summary, "stability_summary")
for c in summary.columns:
    mlflow.log_metric(f"stability_{c.replace(' ', '_')}_mean", summary.loc["mean", c])
    mlflow.log_metric(f"stability_{c.replace(' ', '_')}_std", summary.loc["std", c])
TIMES["6 stability"] = time.perf_counter() - t0

seed 42: scaler equal, Isolation Forest max diff 0.0e+00, cohort stats max diff 0.0e+00
seed 42 XGBoost refit vs saved model on test: max |diff| = 6.654e-02, AP 0.9288 (saved 0.9297), best_iteration 54 (saved 56), band rows 4330 (saved 3887), flag precision 0.9868, recall 0.8310


      best_iteration  test anomalous rows  test anomalous SIMs      AP  flag precision  flag recall  band share  band rows  anomalies in clear  seconds
seed                                                                                                                                                   
1                 57                  377                   28  0.9177          0.9727       0.8515      0.2765       3733                  23      2.0
2                 70                  451                   27  0.9328          0.9875       0.8780      0.1407       1899                  27      2.0
3                 68                  449                   26  0.9464          0.9927       0.9042      0.1967       2656                  15      2.0
4                 14                  201                   15  0.9063          0.0000       0.0000      1.0000      13500                   0      1.7
5                 42                  328                   27  0.9145          0.9892  

## 7. Latency through the service reader

In [9]:
t0 = time.perf_counter()
from app.readers.parquet_reader import ParquetUsageReader
from app.config import SCORE_LOOKBACK_DAYS
reader = ParquetUsageReader(Path("data/synthetic_cdr_daily_agg.csv"))
reader.check()
sims_1k = list(ALL_SIMS[:1000])
day = pd.Timestamp("2025-03-15").date()
lat = []
for _ in range(3):
    ts = time.perf_counter()
    usage = reader.get_daily_usage(sims_1k, day, SCORE_LOOKBACK_DAYS)
    res = rs.score(usage, as_of_dates=day, artifact_dir=ART)
    lat.append(time.perf_counter() - ts)
print(f"1,000 SIMs for one day (reader + score): {min(lat):.2f} s best of 3 {[round(x, 2) for x in lat]}, "
      f"{len(res)} rows scored. Through the API (step 4): about 1.8 s in-process, 2.0 to 2.1 s over HTTP.")
mlflow.log_metric("latency_1000_sims_s", min(lat))
TIMES["7 latency"] = time.perf_counter() - t0

1,000 SIMs for one day (reader + score): 2.07 s best of 3 [2.07, 2.17, 2.23], 1000 rows scored. Through the API (step 4): about 1.8 s in-process, 2.0 to 2.1 s over HTTP.


In [10]:
times = pd.Series(TIMES).round(1).rename("seconds")
print(times.to_string())
log_table(times.to_frame(), "part_times")
mlflow.log_metric("total_seconds", float(times.sum()))
print("MLflow run:", run.info.run_id)
mlflow.end_run()

0 data               6.6
1 pipeline check    14.1
2 row level          0.2
3 stages             0.4
4 SIM level          0.2
5 trade-off          0.0
6 stability         18.4
7 latency            6.7
MLflow run: 11d45da2a7d14f708f550d9bb116300f
